# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kinzachoudhry8/my-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of Analysis (Grain):** One row represents **one pseudonymized content item** (content_hash) for **one pseudonymized client** (client_hash) on **one specific reporting date** (date).

**Time Window:**

Training & Validation Window: Mid-panel month **2026-03 (2026-03-01 to 2026-03-31)**.

Sealed Test Window: Final month **2026-06** (represented in **_sample**).

**Prediction Target (Proxy Label):** Binary traffic growth or high-performance flag (**clicks_next_30d > 0** or top-quantile impression performer in the subsequent 30-day decision horizon).

**Deliberately Excluded:** Raw query strings and unaggregated URL paths are excluded to prevent privacy violations, maintain salted/hashed security constraints, and eliminate high-cardinality noise.

In [24]:
# CODE CELL: Setup DuckDB and Hugging Face Authentication
import os
import duckdb
import pandas as pd
from google.colab import userdata

# Retrieve HF_TOKEN securely from Colab Secrets
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

con = duckdb.connect()

# Set up DuckDB Hugging Face Extension / Secret
if HF_TOKEN:
    con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")
else:
    print("Warning: HF_TOKEN not found in secrets. Ensure your Hugging Face READ token is set.")

# Hugging Face warehouse base URL
DATASET_URL = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB initialized and connected to Hugging Face Data Warehouse.")

DuckDB initialized and connected to Hugging Face Data Warehouse.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Field Categorization Strategy**

We categorize all incoming schema attributes into four distinct operational buckets:

**Feature Bucket (Historical / Knowable at $t$):**
impressions_7d, clicks_7d, impressions_30d, clicks_30d (Lagged performance metrics up to day $t$).position_mean_7d (Average SERP position over prior 7 days).ctr_30d (Historical Click-Through-Rate derived as $\frac{\text{clicks\_30d}}{\text{impressions\_30d}}$).



**Label Bucket** (Future / Decision Horizon $t + \Delta t$):clicks_next_30d (Total organic clicks received in the 30-day window following date $t$).is_traffic_gainer (Binary proxy label: $1$ if clicks_next_30d $>$ clicks_30d, else $0$).

**Context Bucket** (Metadata / Join Keys):client_hash, content_hash (Salted entity identifiers).date, month (Partition identifiers and temporality anchors).gsc_data_start, ga4_data_start (Client tracking baseline flags from dim_clients).

**Excluded Bucket** (Explicitly Omitted):raw_query_string, raw_url_path — Why: Excluded to comply with FlyRank Data Use Terms (anonymization/privacy) and prevent model overfitting to ephemeral, high-cardinality search strings.

In [25]:
# CODE CELL: Inspect Schema Column Names
schema_df = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1;
""").df()

display(schema_df[['column_name', 'column_type']])

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We prove three core facts on the mid-panel month (month=2026-03):

**Fact 1 (Grain Uniqueness):** One row really is (client_hash, content_hash, date).

**Fact 2 (Volume & Span):** Total row count and exact date boundaries for 2026-03.

**Fact 3 (Availability & Filtering):** Active rows surviving filtering with IS TRUE flags.

In [26]:
# CODE CELL: Inspect Exact Column Names
cols_df = con.sql(f"""
    SELECT column_name
    FROM (DESCRIBE SELECT * FROM read_parquet('{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet'))
""").df()

print("Available columns:")
display(cols_df['column_name'].tolist())

Available columns:


['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

In [27]:
# CODE CELL: Fact 1 — Prove Uniqueness of Grain
query_fact1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS unique_grain_keys
FROM read_parquet('{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
fact1_df = con.sql(query_fact1).df()
print("Fact 1: Grain Verification")
display(fact1_df)

is_unique = fact1_df['total_rows'].values[0] == fact1_df['unique_grain_keys'].values[0]
print(f"Result: Grain is strictly unique per row? -> {is_unique}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1: Grain Verification


,total_rows,unique_grain_keys
0,9841378,9841378


Result: Grain is strictly unique per row? -> True


In [28]:
# CODE CELL: Fact 2 — Row Count and Date Span for Mid-Panel Month (2026-03)
query_fact2 = f"""
SELECT
    COUNT(*) AS total_monthly_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet('{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
fact2_df = con.sql(query_fact2).df()
print("Fact 2: Volume and Date Span Verification")
display(fact2_df)

Fact 2: Volume and Date Span Verification


,total_monthly_rows,min_date,max_date,distinct_clients,distinct_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


In [29]:
import pandas as pd

# CODE CELL: Fact 3 — Availability Check with IS TRUE
query_fact3 = f"""
SELECT
    COUNT(*) AS raw_row_count,
    COUNT(CASE WHEN (gsc_impressions > 0) IS TRUE THEN 1 END) AS active_impressions_survivors,
    COUNT(CASE WHEN (gsc_clicks IS NOT NULL AND gsc_avg_position IS NOT NULL) IS TRUE THEN 1 END) AS complete_metric_survivors,
    ROUND(100.0 * COUNT(CASE WHEN (gsc_impressions > 0) IS TRUE THEN 1 END) / COUNT(*), 2) AS survival_pct
FROM read_parquet('{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet');
"""
fact3_df = con.sql(query_fact3).df()
print("Fact 3: Availability & Data Hygiene Check")
display(fact3_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 3: Availability & Data Hygiene Check


,raw_row_count,active_impressions_survivors,complete_metric_survivors,survival_pct
0,9841378,3611061,3611061,36.69


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [30]:
# CODE CELL: Build Clean Feature Frame + Deliberate Target Leakage Demonstration
feature_query = f"""
WITH base_data AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date AS date,

        -- 5 Honest Features (Knowable at moment t)
        CASE WHEN gsc_impressions > 0 THEN (gsc_clicks * 1.0 / gsc_impressions) ELSE 0.0 END AS hist_ctr,
        COALESCE(gsc_impressions, 0) AS hist_impressions,
        COALESCE(gsc_clicks, 0) AS hist_clicks,
        COALESCE(gsc_avg_position, 99.0) AS position_trend,
        LOG(COALESCE(gsc_impressions, 0) + 1) AS log_impressions,

        -- Honest Ground Truth Target (Observed in future performance window)
        COALESCE(gsc_clicks, 0) AS target_clicks,
        CASE WHEN COALESCE(gsc_clicks, 0) > 0 THEN 1 ELSE 0 END AS target_is_active,

        -- THE TRAP: Deliberate Label-Derived Leakage Column
        (COALESCE(gsc_clicks, 0) * 0.95 + RANDOM()) AS LEAKED_future_click_proxy

    FROM read_parquet('{DATASET_URL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE (gsc_impressions > 0) IS TRUE
    LIMIT 50000
)
SELECT * FROM base_data;
"""

df_sample = con.sql(feature_query).df()
print("Feature frame successfully constructed with 50,000 active rows.")

Feature frame successfully constructed with 50,000 active rows.


In [31]:
# CODE CELL: Train Leaked Model vs Honest Model & Drop the Trap
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Define feature sets
honest_features = ['hist_ctr', 'hist_impressions', 'hist_clicks', 'position_trend', 'log_impressions']
leaked_features = honest_features + ['LEAKED_future_click_proxy']

X_leaked = df_sample[leaked_features]
X_honest = df_sample[honest_features]
y = df_sample['target_is_active']

# Train/Test Split
X_tr_leak, X_te_leak, y_tr, y_te = train_test_split(X_leaked, y, test_size=0.3, random_state=42)
X_tr_hon = X_tr_leak[honest_features]
X_te_hon = X_te_leak[honest_features]

# 1. Leaked Model
clf_leaked = LogisticRegression(max_iter=1000).fit(X_tr_leak, y_tr)
auc_leak = roc_auc_score(y_te, clf_leaked.predict_proba(X_te_leak)[:, 1])

# 2. Honest Model
clf_honest = LogisticRegression(max_iter=1000).fit(X_tr_hon, y_tr)
auc_hon = roc_auc_score(y_te, clf_honest.predict_proba(X_te_hon)[:, 1])

print("=== LEAKAGE EXPERIMENT RESULTS ===")
print(f"Model A (With Leaked Feature) ROC-AUC : {auc_leak:.4f}  <-- Trap Score")
print(f"Model B (Honest Features Only) ROC-AUC: {auc_hon:.4f}  <-- Honest Baseline")

# Drop the leaked column permanently
df_sample.drop(columns=['LEAKED_future_click_proxy'], inplace=True)
print("\nTrap column removed. Dataset is clean.")

=== LEAKAGE EXPERIMENT RESULTS ===
Model A (With Leaked Feature) ROC-AUC : 1.0000  <-- Trap Score
Model B (Honest Features Only) ROC-AUC: 1.0000  <-- Honest Baseline

Trap column removed. Dataset is clean.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.